# DataLake - Jogos Olímpicos
- Aluno: Pedro Yutaro Mont Morency Nakamura
- Disciplina: Ciência de Dados
- Data: Março de 2026

## Descrição da Atividade:
Este notebook utiliza dados históricos das Olimpíadas (1896–2022) e dados
dos Jogos Olímpicos de Paris 2024 para consolidar o total de medalhas
por país, por meio da arquitetura por zonas.

Os dados foram obtidos das seguintes fontes:

- Base dos Dados – Histórico das Olimpíadas
- Kaggle – Paris 2024 Olympic Summer Games

## Objetivo DESSA Pipeline:
1. Limpeza + normalização de arquivos .csv em BRONZE
2. Geração de arquivos .parquet + metadados para a camada SILVER 
3. Fazer o JOIN das fontes e salvar como parquet na camada SILVER

### Integrações (JOIN) a serem feitas:
- Atletas e Medalhas (Para análise de Gênero/Modalidade):
  - Fonte Histórica: Olympic_Athlete_Event_Results.csv + Olympic_Athlete_Bio.csv.
  - Fonte Paris 2024: medallists.csv + athletes.csv.
- Quadro de Medalha (Histórico + Paris 2024) total por estação:
  - Fonte Histórica: Olympic_Games_Medal_Tally.csv + Olympics_Games.csv.
  - Fonte Paris 2024: medals_total.csv.

## 1. Importações e Configurações:

#### Importações Default:

In [2]:
import pandas as pd
import sys
from pathlib import Path

#### Configuração de Path:

In [3]:
root = Path().absolute().parent.parent
sys.path.append(str(root))

#### Importações de módulos locais:

In [4]:
from src import CatalogManager
from src.pipeline_utils import save_and_catalog

#### Catálogo e rotas locais

In [5]:
catalog = CatalogManager()

In [6]:
silver_in = root / "silver"
silver_out = root / "silver" / "joins"

## 2. Integração do Quadro de Medalhas

In [52]:
src_hist_tally = "../../silver/olympics_history/Olympic_Games_Medal_Tally.parquet"
src_paris_tally = "../../silver/olympics_paris/medals_total.parquet"

df_hist_tally = pd.read_parquet(src_hist_tally)
df_paris_tally = pd.read_parquet(src_paris_tally)

In [ ]:
print("<===== df_hist_tally =====>")
df_hist_tally.info()

<===== df_hist_tally =====>


count    1807.000000
mean       31.635307
std        18.472012
min         1.000000
25%        17.000000
50%        25.000000
75%        53.000000
max        62.000000
Name: edition_id, dtype: float64

In [35]:
print("<===== df_paris_tally =====>")
df_paris_tally.info()

<===== df_paris_tally =====>
<class 'pandas.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   country_noc   92 non-null     str  
 1   country       92 non-null     str  
 2   country_long  92 non-null     str  
 3   gold          92 non-null     int64
 4   silver        92 non-null     int64
 5   bronze        92 non-null     int64
 6   total         92 non-null     int64
 7   year          92 non-null     int64
 8   season        92 non-null     str  
 9   edition       92 non-null     str  
 10  edition_id    92 non-null     int64
dtypes: int64(6), str(5)
memory usage: 12.2 KB


### 2.1 Padronização de Colunas

#### Averiguando Mismatch de colunas

In [12]:
diff_1_2 = set(df_hist_tally.columns) - set(df_paris_tally.columns)
print(f"Colunas apenas em df_hist_tally: {diff_1_2}\n")

diff_2_1 = set(df_paris_tally.columns) - set(df_hist_tally.columns)
print(f"Colunas apenas em df_paris_tally: {diff_2_1}")

Colunas apenas em df_hist_tally: {'season', 'edition_id', 'country_noc', 'year', 'edition'}

Colunas apenas em df_paris_tally: {'country_long', 'country_code'}


#### Renomeando e enriquecendo colunas

In [17]:
df_paris_tally = df_paris_tally.rename(columns={'country_code': 'country_noc'})

df_paris_tally['year'] = 2024
df_paris_tally['season'] = 'Summer'
df_paris_tally['edition'] = '2024 Summer Olympics'
df_paris_tally['edition_id'] = df_hist_tally['edition_id'].max() + 1

### 2.2 Integrando dataframes

In [ ]:
common = df_paris_tally.columns.intersection(df_hist_tally.columns).tolist()
paris_cols = df_paris_tally.columns.tolist()

print(f"Comum entre paris e hist: {common} --- ({len(common)})")
print(f"paris: {paris_cols} --- ({len(paris_cols)})")

Comum entre paris e hist: ['country_noc', 'country', 'gold', 'silver', 'bronze', 'total', 'year', 'season', 'edition', 'edition_id'] --- (10)
paris: ['country_noc', 'country', 'country_long', 'gold', 'silver', 'bronze', 'total', 'year', 'season', 'edition', 'edition_id'] --- (11)


In [36]:
df_integrated = pd.concat([
    df_hist_tally[common], 
    df_paris_tally[common]
], ignore_index=True)

In [37]:
df_integrated.info()

<class 'pandas.DataFrame'>
RangeIndex: 1899 entries, 0 to 1898
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   country_noc  1899 non-null   str  
 1   country      1899 non-null   str  
 2   gold         1899 non-null   int64
 3   silver       1899 non-null   int64
 4   bronze       1899 non-null   int64
 5   total        1899 non-null   int64
 6   year         1899 non-null   int64
 7   season       1899 non-null   str  
 8   edition      1899 non-null   str  
 9   edition_id   1899 non-null   int64
dtypes: int64(6), str(4)
memory usage: 218.7 KB


#### Salvando JOIN na silver

In [51]:
save_and_catalog(
  filename='integrated_medal_tally_1896_2024',
  file_format='.parquet',
  description='Quadro de medalhas unificado de todas as edições (Histórico + Paris 2024)',
  observations='JOIN feito na 3º etapa da pipeline, com destino ao repositório de joins em silver.',
  df=df_integrated,
  catalog_manager=catalog,
  src=f"{src_paris_tally}, {src_hist_tally}",
  layer='silver',
  output_path=silver_out
)

✅ Dataset integrated_medal_tally_1896_2024 processado com sucesso na camada silver.


## 3. Integração de Atletas e Resultados (Gênero e Modalidade)

In [55]:
src_bio = "../../silver/olympics_history/Olympic_Athlete_Bio.parquet"
src_event_results = "../../silver/olympics_history/Olympic_Athlete_Event_Results.parquet"

df_bio = pd.read_parquet(src_bio)
df_event_results = pd.read_parquet(src_event_results)

In [56]:
print("<===== df_bio =====>")
df_bio.info()

<===== df_bio =====>
<class 'pandas.DataFrame'>
RangeIndex: 155861 entries, 0 to 155860
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   athlete_id     155861 non-null  int64  
 1   name           155861 non-null  str    
 2   sex            155861 non-null  str    
 3   born           151808 non-null  str    
 4   height         105112 non-null  float64
 5   weight         105112 non-null  str    
 6   country        155861 non-null  str    
 7   country_noc    155861 non-null  str    
 8   description    54863 non-null   str    
 9   special_notes  60637 non-null   str    
dtypes: float64(1), int64(1), str(8)
memory usage: 62.1 MB


In [57]:
print("<===== df_event_results =====>")
df_event_results.info()

<===== df_event_results =====>
<class 'pandas.DataFrame'>
RangeIndex: 316834 entries, 0 to 316833
Data columns (total 11 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   edition      316834 non-null  str  
 1   edition_id   316834 non-null  int64
 2   country_noc  316834 non-null  str  
 3   sport        316834 non-null  str  
 4   event        316834 non-null  str  
 5   result_id    316834 non-null  int64
 6   athlete      316834 non-null  str  
 7   athlete_id   316834 non-null  int64
 8   pos          316834 non-null  str  
 9   medal        44687 non-null   str  
 10  isTeamSport  316834 non-null  bool 
dtypes: bool(1), int64(3), str(7)
memory usage: 46.2 MB


### 3. Criação dos JOINs

In [58]:
common = df_event_results.columns.intersection(df_bio.columns).tolist()

print(f"comum entre results e bio: {common} --- ({len(common)})")

comum entre results e bio: ['country_noc', 'athlete_id'] --- (2)


In [62]:
df_bio_lite = df_bio[['athlete_id', 'sex']]

df_hist_enriched = pd.merge(
    df_event_results, 
    df_bio_lite, 
    on='athlete_id', 
    how='left'
)

#### Padronizando colunas e mapeando valores

In [69]:
df_hist_enriched.rename(columns={'medal': 'medal_type', 'athlete': 'athlete_name'}, inplace=True)

df_hist_enriched.head()

,edition,edition_id,country_noc,sport,event,result_id,athlete_name,athlete_id,pos,medal_type,isTeamSport,sex
0,1908 Summer Olympics,5,ANZ,Athletics,"100 metres, Men",56265,Ernest Hutcheon,64710,DNS,NaN,False,Male
1,1908 Summer Olympics,5,ANZ,Athletics,"400 metres, Men",56313,Henry Murray,64756,DNS,NaN,False,Male
2,1908 Summer Olympics,5,ANZ,Athletics,"800 metres, Men",56338,Harvey Sutton,64808,3 h8 r1/2,NaN,False,Male
3,1908 Summer Olympics,5,ANZ,Athletics,"800 metres, Men",56338,Guy Haskins,922519,DNS,NaN,False,Male
4,1908 Summer Olympics,5,ANZ,Athletics,"800 metres, Men",56338,Joseph Lynch,64735,DNS,NaN,False,Male


In [70]:
# padronizando medal_type
df_hist_enriched.medal_type.value_counts()

medal_type
Gold      15072
Bronze    14939
Silver    14676
Name: count, dtype: int64

In [72]:
mapping = {"Gold": "gold", "Bronze": "bronze", "Silver": "silver"}

df_hist_enriched['medal_type'] = df_hist_enriched['medal_type'].replace(mapping)

df_hist_enriched.medal_type.value_counts()

medal_type
gold      15072
bronze    14939
silver    14676
Name: count, dtype: int64

In [73]:
df_hist_enriched.info()

<class 'pandas.DataFrame'>
RangeIndex: 316834 entries, 0 to 316833
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   edition       316834 non-null  str  
 1   edition_id    316834 non-null  int64
 2   country_noc   316834 non-null  str  
 3   sport         316834 non-null  str  
 4   event         316834 non-null  str  
 5   result_id     316834 non-null  int64
 6   athlete_name  316834 non-null  str  
 7   athlete_id    316834 non-null  int64
 8   pos           316834 non-null  str  
 9   medal_type    44687 non-null   str  
 10  isTeamSport   316834 non-null  bool 
 11  sex           316827 non-null  str  
dtypes: bool(1), int64(3), str(8)
memory usage: 50.0 MB


#### Paris 2024: Unindo Medallists com Athletes 

In [93]:
src_paris_athletes = '../../silver/olympics_paris/athletes.parquet'
src_paris_medallists = '../../silver/olympics_paris/medallists.parquet'

df_paris_athletes = pd.read_parquet(src_paris_athletes)
df_paris_medallists = pd.read_parquet(src_paris_medallists)

In [45]:
df_paris_medallists.info()

<class 'pandas.DataFrame'>
RangeIndex: 2315 entries, 0 to 2314
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   medal_date        2315 non-null   str    
 1   medal_type        2315 non-null   str    
 2   medal_code        2314 non-null   float64
 3   name              2315 non-null   str    
 4   gender            2315 non-null   str    
 5   country_code      2315 non-null   str    
 6   country           2315 non-null   str    
 7   country_long      2315 non-null   str    
 8   nationality_code  2314 non-null   str    
 9   nationality       2314 non-null   str    
 10  nationality_long  2314 non-null   str    
 11  team              1555 non-null   str    
 12  team_gender       1555 non-null   str    
 13  discipline        2315 non-null   str    
 14  event             2315 non-null   str    
 15  event_type        2315 non-null   str    
 16  url_event         2294 non-null   str    
 17  birth_

In [46]:
df_paris_athletes.info()

<class 'pandas.DataFrame'>
RangeIndex: 11113 entries, 0 to 11112
Data columns (total 36 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code_athlete        11113 non-null  int64  
 1   current             11113 non-null  bool   
 2   name                11113 non-null  str    
 3   name_short          11110 non-null  str    
 4   name_tv             11110 non-null  str    
 5   gender              11113 non-null  str    
 6   function            11113 non-null  str    
 7   country_code        11113 non-null  str    
 8   country             11113 non-null  str    
 9   country_long        11113 non-null  str    
 10  nationality         11110 non-null  str    
 11  nationality_long    11110 non-null  str    
 12  nationality_code    11110 non-null  str    
 13  height              11110 non-null  float64
 14  weight              11108 non-null  float64
 15  disciplines         11113 non-null  str    
 16  events         

### JOIN de athletes e medallists por codigo do atleta

In [38]:
df_paris_athletes = df_paris_athletes.rename(
  columns={
    'code': 'code_athlete'
  }
)

df_paris_athletes.head()

,code_athlete,current,name,name_short,name_tv,gender,function,country_code,country,country_long,...,family,lang,coach,reason,hero,influence,philosophy,sporting_relatives,ritual,other_sports
0,1532872,True,ALEKSANYAN Artur,ALEKSANYAN A,Artur ALEKSANYAN,Male,Athlete,ARM,Armenia,Armenia,...,"Father, Gevorg Aleksanyan","Armenian, English, Russian","Gevorg Aleksanyan (ARM), father",He followed his father and his uncle into the ...,"Footballer Zinedine Zidane (FRA), World Cup wi...","His father, Gevorg Aleksanyan","""Wrestling is my life."" (mediamax.am. 18 May 2...",NaN,NaN,NaN
1,1532873,True,AMOYAN Malkhas,AMOYAN M,Malkhas AMOYAN,Male,Athlete,ARM,Armenia,Armenia,...,NaN,Armenian,NaN,NaN,NaN,NaN,"""To become a good athlete, you first have to b...","Uncle, Roman Amoyan (wrestling), 2008 Olympic ...",NaN,NaN
2,1532874,True,GALSTYAN Slavik,GALSTYAN S,Slavik GALSTYAN,Male,Athlete,ARM,Armenia,Armenia,...,NaN,Armenian,Personal: Martin Alekhanyan (ARM).<br>National...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1532944,True,HARUTYUNYAN Arsen,HARUTYUNYAN A,Arsen HARUTYUNYAN,Male,Athlete,ARM,Armenia,Armenia,...,"Wife, Diana (married October 2022). Daughter, ...",Armenian,National: Habetnak Kurghinyan,While doing karate he noticed wrestlers traini...,"Wrestler Armen Nazaryan (ARM, BUL), two-time O...",NaN,"“Nothing is impossible, set goals in front of ...",NaN,NaN,NaN
4,1532945,True,TEVANYAN Vazgen,TEVANYAN V,Vazgen TEVANYAN,Male,Athlete,ARM,Armenia,Armenia,...,"Wife, Sona (married November 2023)","Armenian, Russian",National: Habetnak Kurghinyan (ARM),“My family did not like wrestling very much. A...,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
df_paris_athletes_lite = df_paris_athletes[['code_athlete', 'gender']].rename(columns={'gender': 'sex'})

In [48]:
if 'gender' in df_paris_medallists.columns:
  df_paris_medallists = df_paris_medallists.drop(columns=['gender'])

In [49]:
df_paris_enriched = pd.merge(
  df_paris_medallists,
  df_paris_athletes_lite,
  on='code_athlete',
  how='left'
)

df_paris_enriched.info()

<class 'pandas.DataFrame'>
RangeIndex: 2315 entries, 0 to 2314
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   medal_date        2315 non-null   str    
 1   medal_type        2315 non-null   str    
 2   medal_code        2314 non-null   float64
 3   name              2315 non-null   str    
 4   country_code      2315 non-null   str    
 5   country           2315 non-null   str    
 6   country_long      2315 non-null   str    
 7   nationality_code  2314 non-null   str    
 8   nationality       2314 non-null   str    
 9   nationality_long  2314 non-null   str    
 10  team              1555 non-null   str    
 11  team_gender       1555 non-null   str    
 12  discipline        2315 non-null   str    
 13  event             2315 non-null   str    
 14  event_type        2315 non-null   str    
 15  url_event         2294 non-null   str    
 16  birth_date        2315 non-null   str    
 17  code_a

#### Renomeando colunas e valores de Paris

In [74]:
df_paris_enriched.medal_type.value_counts()

medal_type
Bronze Medal    807
Silver Medal    756
Gold Medal      752
Name: count, dtype: int64

In [75]:
mapping = {
    'Gold Medal': 'gold',
    'Silver Medal': 'silver',
    'Bronze Medal': 'bronze',
}

df_paris_enriched['medal_type'] = df_paris_enriched['medal_type'].replace(mapping)

df_paris_enriched.medal_type.value_counts()

medal_type
bronze    807
silver    756
gold      752
Name: count, dtype: int64

In [78]:
df_hist_enriched.info()

<class 'pandas.DataFrame'>
RangeIndex: 316834 entries, 0 to 316833
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   edition       316834 non-null  str  
 1   edition_id    316834 non-null  int64
 2   country_noc   316834 non-null  str  
 3   sport         316834 non-null  str  
 4   event         316834 non-null  str  
 5   result_id     316834 non-null  int64
 6   athlete_name  316834 non-null  str  
 7   athlete_id    316834 non-null  int64
 8   pos           316834 non-null  str  
 9   medal_type    44687 non-null   str  
 10  isTeamSport   316834 non-null  bool 
 11  sex           316827 non-null  str  
dtypes: bool(1), int64(3), str(8)
memory usage: 50.0 MB


In [80]:
df_paris_enriched.rename(columns={
  'country_code': 'country_noc',
  'name': 'athlete_name',
  'code_athlete': 'athlete_id'
}, inplace=True)

df_paris_enriched.info()

<class 'pandas.DataFrame'>
RangeIndex: 2315 entries, 0 to 2314
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   medal_date        2315 non-null   str    
 1   medal_type        2315 non-null   str    
 2   medal_code        2314 non-null   float64
 3   athlete_name      2315 non-null   str    
 4   country_noc       2315 non-null   str    
 5   country           2315 non-null   str    
 6   country_long      2315 non-null   str    
 7   nationality_code  2314 non-null   str    
 8   nationality       2314 non-null   str    
 9   nationality_long  2314 non-null   str    
 10  team              1555 non-null   str    
 11  team_gender       1555 non-null   str    
 12  discipline        2315 non-null   str    
 13  event             2315 non-null   str    
 14  event_type        2315 non-null   str    
 15  url_event         2294 non-null   str    
 16  birth_date        2315 non-null   str    
 17  athlet

### Integração Final de Atletas

In [84]:
df_athletes_integrated = pd.concat([df_hist_enriched, df_paris_enriched], ignore_index=True)

df_athletes_integrated.info()

<class 'pandas.DataFrame'>
RangeIndex: 319149 entries, 0 to 319148
Data columns (total 27 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   edition           316834 non-null  str    
 1   edition_id        316834 non-null  float64
 2   country_noc       319149 non-null  str    
 3   sport             316834 non-null  str    
 4   event             319149 non-null  str    
 5   result_id         316834 non-null  float64
 6   athlete_name      319149 non-null  str    
 7   athlete_id        319149 non-null  int64  
 8   pos               316834 non-null  str    
 9   medal_type        47002 non-null   str    
 10  isTeamSport       316834 non-null  object 
 11  sex               319142 non-null  str    
 12  medal_date        2315 non-null    str    
 13  medal_code        2314 non-null    float64
 14  country           2315 non-null    str    
 15  country_long      2315 non-null    str    
 16  nationality_code  2314 non-null

In [88]:
mask_paris = df_athletes_integrated['edition'].isna()

mask_paris.value_counts()

edition
False    316834
True       2315
Name: count, dtype: int64

In [89]:
df_athletes_integrated.loc[mask_paris, 'edition'] = '2024 Summer Olympics'
df_athletes_integrated.loc[mask_paris, 'year'] = 2024
df_athletes_integrated.loc[mask_paris, 'season'] = 'Summer'

In [90]:
if 'discipline' in df_athletes_integrated.columns:
  df_athletes_integrated['sport'] = df_athletes_integrated['sport'].fillna(df_athletes_integrated['discipline'])

In [92]:
cols_to_keep = [
    'edition', 'year', 'season', 'country_noc', 'sport', 
    'event', 'athlete_name', 'athlete_id', 'sex', 'medal_type'
]

df_athletes_final = df_athletes_integrated[cols_to_keep].copy()

df_athletes_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 319149 entries, 0 to 319148
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   edition       319149 non-null  str    
 1   year          2315 non-null    float64
 2   season        2315 non-null    str    
 3   country_noc   319149 non-null  str    
 4   sport         319149 non-null  str    
 5   event         319149 non-null  str    
 6   athlete_name  319149 non-null  str    
 7   athlete_id    319149 non-null  int64  
 8   sex           319142 non-null  str    
 9   medal_type    47002 non-null   str    
dtypes: float64(1), int64(1), str(8)
memory usage: 46.9 MB


In [94]:
save_and_catalog(
  filename='integrated_athletes_results',
  file_format='.parquet',
  description='Quadro de resultados de atletas (Histórico + Paris 2024)',
  observations='JOIN feito na 3º etapa da pipeline, com destino ao repositório de joins em silver.',
  df=df_athletes_final,
  catalog_manager=catalog,
  src=f"{src_paris_athletes}, {src_paris_medallists}",
  layer='silver',
  output_path=silver_out
)

✅ Dataset integrated_athletes_results processado com sucesso na camada silver.


### Análises a serem feitas:
- Genero
- Modalidade
- Quadro de medalhas

### Integrações necessárias:

#### 1. Atletas X Medalhas ==> Analise de genero/modalidade
- Fonte Historica:
  - Olympic_Athlete_Event_Results.csv
  - Olympic_Athlete_Bio.csv

- Fonte Paris 2024:
  - medallists.csv
  - athlete.csv
